# VISCERA: original dataset → conditional shape model → checked variations

This notebook creates original procedural liver/gallbladder samples, trains a real
compact conditional rectified flow, compares conditional PCA and nearest-design
baselines, evaluates an explicit within-generator test split, and packages weights,
reports and numerical mechanics trials. **The anatomy is unreviewed synthetic
design. No tissue or population validation is claimed.** A trained model samples
construction parameters; the supplied procedural decoder generates the 3D mesh.
The default seed reproduces the development pilot; its original test rows have
already been observed during development. Use new seeds and freeze model choices
before a new untouched audit; rerunning this notebook does not reset that history.

Upload the source ZIP produced by the project, or choose an existing checkout.
No paid service, fixed repository URL, pre-existing Drive mount or A100 is assumed.
Select a GPU in Colab if available; CPU also works. Read every cell before Run all.
The default pilot is intentionally small; enlarge only after reviewing failures.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from pathlib import Path

SOURCE_MODE = "upload"  # "upload" or "existing"
EXISTING_PROJECT = "/content/viscera"
INSTALL_DEPENDENCIES = True
DATASET_COUNT = 128
RESOLUTION = "medium"
TRAIN_STEPS = 600
SEED = 20261006
RUN_ID = "adult-design-shape-pilot-v1"  # change for a new experiment; keep to resume
USE_DRIVE = False
DRIVE_SUBDIR = "viscera-checkpoints"
RUN_MECHANICS = True
DOWNLOAD_RESULTS = True

try:
    from google.colab import drive, files

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
if SOURCE_MODE == "upload":
    if not IN_COLAB:
        raise RuntimeError("Outside Colab choose SOURCE_MODE='existing' and set EXISTING_PROJECT")
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Upload exactly one VISCERA source ZIP")
    source_root = Path(tempfile.mkdtemp(prefix="viscera-source-", dir="/content"))
    with zipfile.ZipFile(archives[0]) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)) or "MANIFEST.json" not in names:
            raise ValueError("Source archive requires unique paths and MANIFEST.json")
        total = sum(info.file_size for info in archive.infolist())
        if total > 2_000_000_000:
            raise ValueError("Source archive exceeds the 2 GB expanded-size limit")
        for info in archive.infolist():
            target = (source_root / info.filename).resolve()
            if (
                not target.is_relative_to(source_root.resolve())
                or (info.external_attr >> 16) & 0o170000 == 0o120000
            ):
                raise ValueError("Unsafe archive path or symlink")
        manifest = json.loads(archive.read("MANIFEST.json"))
        if set(manifest["files"]) != set(names) - {"MANIFEST.json"}:
            raise ValueError("Source archive contents disagree with their manifest")
        for name, expected in manifest["files"].items():
            with archive.open(name) as source:
                if hashlib.file_digest(source, "sha256").hexdigest() != expected:
                    raise ValueError("Source checksum mismatch: " + name)
        archive.extractall(source_root)
    candidates = [
        p.parent
        for p in source_root.rglob("pyproject.toml")
        if (p.parent / "py/viscera_core/geometry.py").exists()
    ]
    if len(candidates) != 1:
        raise ValueError("Archive must contain exactly one VISCERA project")
    PROJECT = candidates[0].resolve()
elif SOURCE_MODE == "existing":
    PROJECT = Path(EXISTING_PROJECT).expanduser().resolve()
else:
    raise ValueError("SOURCE_MODE must be upload or existing")
if not (PROJECT / "py/viscera_core/geometry.py").exists():
    raise FileNotFoundError("Project geometry source is missing")
os.chdir(PROJECT)
if INSTALL_DEPENDENCIES:
    requirements = PROJECT / "notebooks/requirements-colab.txt"
    if not requirements.exists():
        raise FileNotFoundError("Pinned Colab requirements missing from source bundle")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--require-hashes", "-r", str(requirements)],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(PROJECT)], check=True
    )
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / "py"))
# Child training/evaluation processes need the same source roots when using an
# existing environment without an editable package installation.
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(PROJECT), str(PROJECT / "py")]
    + [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p]
)
print("Project:", PROJECT)

In [ ]:
import numpy as np
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK = (Path("/content/viscera-work") if IN_COLAB else PROJECT / "artifacts/notebook") / RUN_ID
WORK.mkdir(parents=True, exist_ok=True)
MODEL_DIR = WORK / "model"
DATA_DIR = WORK / "dataset"
REMOTE = None
if USE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Drive staging is available only in Colab; local results stay in WORK")
    drive.mount("/content/drive")
    REMOTE = Path("/content/drive/MyDrive") / DRIVE_SUBDIR / RUN_ID
    REMOTE.mkdir(parents=True, exist_ok=True)
    if (REMOTE / "checkpoint.pt").exists() and not (MODEL_DIR / "checkpoint.pt").exists():
        MODEL_DIR.mkdir(parents=True, exist_ok=True)
        for name in ("checkpoint.pt", "checkpoint.pt.sha256.json"):
            shutil.copy2(REMOTE / name, MODEL_DIR / name)
probe = {
    "device": DEVICE,
    "torch": torch.__version__,
    "python": sys.version,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "vram_bytes": torch.cuda.get_device_properties(0).total_memory if DEVICE == "cuda" else None,
    "free_disk_bytes": shutil.disk_usage(WORK).free,
}
(WORK / "hardware.json").write_text(json.dumps(probe, indent=2))
print(json.dumps(probe, indent=2))

## Original data and strict split files

The release keeps each seed and its descriptions in one split. All samples use
one authored recipe, so the benchmark measures within-generator interpolation.
It cannot establish independent-design or biological generalization. The geometry
checks do not replace expert review. The trainer reads only train/validation;
the explicit evaluation cell below is the only model-scoring test access.

In [ ]:
from viscera_core.dataset import create_dataset

release = create_dataset(
    DATA_DIR,
    count=DATASET_COUNT,
    seed=SEED,
    resolution=RESOLUTION,
    sampler_profile="adult_design_v1",
)
print(
    json.dumps(
        {k: release[k] for k in ("attempted", "accepted", "rejected", "split_counts", "profile")},
        indent=2,
    )
)
if not all(release["split_counts"].get(k, 0) > 0 for k in ("train", "val", "test")):
    raise RuntimeError("Insufficient accepted samples in a split; inspect failures before training")

In [ ]:
command = [
    sys.executable,
    "-m",
    "pipelines.train.viscera_train.shape",
    "train",
    "--data",
    str(DATA_DIR),
    "--out",
    str(MODEL_DIR),
    "--steps",
    str(TRAIN_STEPS),
    "--seed",
    str(SEED),
    "--device",
    DEVICE,
]
if (MODEL_DIR / "checkpoint.pt").exists():
    command.append("--resume")


def persist_checkpoint():
    if REMOTE is None or not (MODEL_DIR / "checkpoint.pt").exists():
        return
    checkpoint = MODEL_DIR / "checkpoint.pt"
    try:
        expected = json.loads((MODEL_DIR / "checkpoint.pt.sha256.json").read_text())["sha256"]
        payload = checkpoint.read_bytes()
        if hashlib.sha256(payload).hexdigest() != expected:
            return  # atomic local checkpoint may be newer than its sidecar; try next poll
        temporary = REMOTE / "checkpoint.pt.uploading"
        temporary.write_bytes(payload)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != expected:
            raise IOError("Drive read-back checksum failed")
        temporary.replace(REMOTE / "checkpoint.pt")
        (REMOTE / "checkpoint.pt.sha256.json").write_text(json.dumps({"sha256": expected}))
    except FileNotFoundError:
        return


started = time.perf_counter()
process = subprocess.Popen(command, cwd=PROJECT)
try:
    while process.poll() is None:
        persist_checkpoint()
        time.sleep(3)
except BaseException:
    process.terminate()
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise
finally:
    persist_checkpoint()
if process.returncode != 0:
    raise RuntimeError("Training failed; inspect output and preserved checkpoint")
training = json.loads((MODEL_DIR / "training_report.json").read_text())
print(
    {
        "status": training["status"],
        "steps": training["step"],
        "wall_time_s": training["wall_time_s"],
        "parameter_count": training["parameter_count"],
        "peak_gpu_bytes": training["peak_gpu_bytes"],
    }
)

## Frozen evaluation and sampled variants

Compare the learned model against conditional Gaussian/PCA and nearest-design
retrieval. Rejections count in the denominator. Control-space MMD is **not** a
surface-anatomy metric. Inspect per-condition measured errors, fixed-condition
diversity, and failure cases. This cell does not retrain or select checkpoints.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pipelines.train.viscera_train.shape",
        "evaluate",
        "--data",
        str(DATA_DIR),
        "--model",
        str(MODEL_DIR / "model.pt"),
        "--split",
        "test",
        "--out",
        str(WORK / "test_evaluation.json"),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pipelines.train.viscera_train.shape",
        "sample",
        "--model",
        str(MODEL_DIR / "model.pt"),
        "--conditions",
        "{}",
        "--count",
        "12",
        "--out",
        str(WORK / "sampled_variants.json"),
        "--scene-dir",
        str(WORK / "generated-scenes"),
    ],
    check=True,
)
evaluation = json.loads((WORK / "test_evaluation.json").read_text())
for method, result in evaluation["methods"].items():
    print(
        method,
        "accepted",
        result["decoded_accepted"],
        "/",
        result["decoded_attempted"],
        "control-space MMD",
        result["parameter_mmd"],
        "coverage",
        result["parameter_coverage"],
    )

In [ ]:
if RUN_MECHANICS:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "viscera_core.mechanics",
            "--out",
            str(WORK / "mechanics"),
            "--resolutions",
            "3,4,5",
        ],
        check=True,
    )
    print(
        "Mechanics is numerical polynomial Eq.13 block-patch verification, not organ T1 contact or tissue validation."
    )

In [ ]:
import matplotlib.pyplot as plt
from viscera_core.geometry import generate_scene

samples = json.loads((WORK / "sampled_variants.json").read_text())["samples"]
accepted = [row for row in samples if row["accepted"]][:6]
fig = plt.figure(figsize=(14, 5))
for index, row in enumerate(accepted):
    scene = generate_scene(
        {"seed": SEED + index, "parameters": row["parameters"]}, resolution=RESOLUTION
    )
    ax = fig.add_subplot(2, 3, index + 1, projection="3d")
    organ = scene["organs"][0]
    vertices = np.asarray(organ["positions"]).reshape(-1, 3)
    faces_array = np.asarray(organ["indices"]).reshape(-1, 3)
    ax.plot_trisurf(
        vertices[:, 0],
        vertices[:, 1],
        vertices[:, 2],
        triangles=faces_array,
        color="#aa594e",
        linewidth=0,
        alpha=0.95,
    )
    ax.set_box_aspect(np.ptp(vertices, axis=0))
    ax.set_axis_off()
    ax.set_title(f"Unreviewed synthetic variant {index + 1}")
if not accepted:
    fig.text(0.1, 0.5, "No sampled candidate passed; inspect failure records.")
fig.tight_layout()
fig.savefig(WORK / "sampled_variants.png", dpi=130)
plt.show()

In [ ]:
bundle_base = WORK.parent / (RUN_ID + "-results")
archive = Path(shutil.make_archive(str(bundle_base), "zip", WORK))
print("Result archive:", archive, "bytes:", archive.stat().st_size)
if REMOTE is not None:
    shutil.copy2(archive, REMOTE / archive.name)
    if (
        hashlib.sha256((REMOTE / archive.name).read_bytes()).hexdigest()
        != hashlib.sha256(archive.read_bytes()).hexdigest()
    ):
        raise IOError("Result archive Drive checksum mismatch")
if DOWNLOAD_RESULTS and IN_COLAB:
    files.download(str(archive))
print(
    "Weights, decoder source identity, dataset release, baseline comparison, failures, model card and numerical mechanics are packaged."
)